<a href="https://colab.research.google.com/github/mfza0/ecommerce-revenue-analytics/blob/main/rev_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3
import numpy as np
import pandas as pd

In [2]:
url = "https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv"
print("1. Downloading real UCI Online Retail dataset...")
raw_df = pd.read_csv(url)
print(f"   Raw Rows Loaded: {len(raw_df):,}")

1. Downloading real UCI Online Retail dataset...
   Raw Rows Loaded: 541,909


In [3]:
#raw_df.describe()
#raw_df.head()
nulls = (raw_df.isnull().sum() / len(raw_df)) * 100


In [4]:
df = raw_df.copy()
df.columns = [
    'invoice_no',
    'stock_code',
    'description',
    'quantity',
    'invoice_date',
    'unit_price',
    'customer_id',
    'country',
]

In [5]:
initial_rows = len(df)
df = df.drop_duplicates()
print(f"2. Removed {initial_rows - len(df):,} exact duplicate rows.")

2. Removed 5,268 exact duplicate rows.


In [6]:
#df['stock_code'].unique()
invalid_codes = [
    'POST',
    'D',
    'M',
    'BANK CHARGES',
    'PADS',
    'DOT',
    'CRUK',
    'C2',
    'AMAZONFEE',
]
df = df[~df['stock_code'].isin(invalid_codes)]
df = df[df['unit_price'] > 0]

In [7]:
df['order_status'] = np.where(
    df['invoice_no'].astype(str).str.startswith('C') | (df['quantity'] < 0),
    'Canceled/Returned',
    'Delivered',
)


In [8]:
# 4. Handle Missing Values:
#    - Keep guest checkouts labeled as 'GUEST' for store-level revenue, but separate for RFM
df['description'] = df['description'].fillna('Unknown Product').str.strip()
df['is_registered'] = np.where(df['customer_id'].isna(), 0, 1)
df['customer_id'] = (
    df['customer_id'].fillna(0).astype(int).astype(str).replace('0', 'GUEST')
)

In [9]:
# 5. Parse Timestamps & Calculate Financials
df['invoice_date'] = pd.to_datetime(df['invoice_date'])
df['invoice_year_month'] = df['invoice_date'].dt.strftime('%Y-%m')
df['abs_quantity'] = np.abs(df['quantity'])
df['gross_amount'] = np.round(df['abs_quantity'] * df['unit_price'], 2)

In [10]:
# Net Revenue is positive for Delivered, 0 for Canceled; Leakage captures Canceled value
df['net_revenue'] = np.where(
    df['order_status'] == 'Delivered', df['gross_amount'], 0.0
)
df['revenue_leakage'] = np.where(
    df['order_status'] == 'Canceled/Returned', df['gross_amount'], 0.0
)

print(f"3. Cleaned Fact Table Ready: {len(df):,} valid rows.")

3. Cleaned Fact Table Ready: 531,300 valid rows.


In [11]:
# A. Customer RFM Dimension (Registered Customers with Delivered Orders)
delivered_reg = df[
    (df['order_status'] == 'Delivered') & (df['customer_id'] != 'GUEST')
]
snapshot_date = delivered_reg['invoice_date'].max() + pd.Timedelta(days=1)

dim_customers_rfm = (
    delivered_reg.groupby(['customer_id', 'country'])
    .agg(
        recency_days=('invoice_date', lambda x: (snapshot_date - x.max()).days),
        frequency_orders=('invoice_no', 'nunique'),
        monetary_value=('net_revenue', 'sum'),
        first_purchase=('invoice_date', 'min'),
        last_purchase=('invoice_date', 'max'),
    )
    .reset_index()
)
dim_customers_rfm['monetary_value'] = dim_customers_rfm['monetary_value'].round(
    2
)

In [12]:
# Quartile Scoring (1 = Lowest, 4 = Highest)
dim_customers_rfm['R_Score'] = pd.qcut(
    dim_customers_rfm['recency_days'], q=4, labels=[4, 3, 2, 1]
).astype(int)
dim_customers_rfm['F_Score'] = pd.qcut(
    dim_customers_rfm['frequency_orders'].rank(method='first'),
    q=4,
    labels=[1, 2, 3, 4],
).astype(int)
dim_customers_rfm['M_Score'] = pd.qcut(
    dim_customers_rfm['monetary_value'], q=4, labels=[1, 2, 3, 4]
).astype(int)
dim_customers_rfm['RFM_Index'] = (
    dim_customers_rfm['R_Score']
    + dim_customers_rfm['F_Score']
    + dim_customers_rfm['M_Score']
)

dim_customers_rfm['customer_segment'] = np.select(
    [
        dim_customers_rfm['RFM_Index'] >= 10,
        dim_customers_rfm['RFM_Index'] >= 8,
        dim_customers_rfm['RFM_Index'] >= 6,
        dim_customers_rfm['R_Score'] == 1,
    ],
    ['Champions', 'Loyal Customers', 'Potential Loyalists', 'At-Risk / Churned'],
    default='Need Attention',
)

In [13]:
# B. Product Dimension Table (Deduplicated Stock Codes)
dim_products = (
    df.groupby('stock_code')
    .agg(
        product_name=('description', 'first'),
        avg_unit_price=('unit_price', lambda x: round(x.mean(), 2)),
    )
    .reset_index()
)

print(
    f"4. Segmented {len(dim_customers_rfm):,} unique customers & {len(dim_products):,} products."
)

4. Segmented 4,342 unique customers & 3,929 products.


In [14]:
conn = sqlite3.connect('uci_retail_star_schema.db')

df_sql = df.copy()
df_sql['invoice_date'] = df_sql['invoice_date'].dt.strftime('%Y-%m-%d %H:%M:%S')
dim_customers_sql = dim_customers_rfm.copy()
dim_customers_sql['first_purchase'] = dim_customers_sql[
    'first_purchase'
].dt.strftime('%Y-%m-%d')
dim_customers_sql['last_purchase'] = dim_customers_sql[
    'last_purchase'
].dt.strftime('%Y-%m-%d')

df_sql.to_sql('fact_transactions', conn, if_exists='replace', index=False)
dim_customers_sql.to_sql(
    'dim_customers_rfm', conn, if_exists='replace', index=False
)
dim_products.to_sql('dim_products', conn, if_exists='replace', index=False)

3929

In [15]:
# --- SQL 1: Month-over-Month (MoM) Net Revenue & Order Growth (CTE + LAG) ---
sql_1_mom = """
WITH Monthly_KPIs AS (
    SELECT
        invoice_year_month AS sales_month,
        COUNT(DISTINCT CASE WHEN order_status = 'Delivered' THEN invoice_no END) AS delivered_orders,
        ROUND(SUM(net_revenue), 2) AS net_revenue,
        ROUND(SUM(revenue_leakage), 2) AS canceled_leakage
    FROM fact_transactions
    GROUP BY 1
)
SELECT
    sales_month,
    delivered_orders,
    net_revenue,
    canceled_leakage,
    LAG(net_revenue) OVER (ORDER BY sales_month) AS prev_month_rev,
    ROUND(
        ((net_revenue - LAG(net_revenue) OVER (ORDER BY sales_month)) /
        LAG(net_revenue) OVER (ORDER BY sales_month)) * 100, 2
    ) AS mom_growth_pct
FROM Monthly_KPIs
ORDER BY sales_month;
"""

In [16]:
print("\n--- SQL 1: MONTH-OVER-MONTH REVENUE & LEAKAGE ---")
print(pd.read_sql(sql_1_mom, conn).to_string(index=False))


--- SQL 1: MONTH-OVER-MONTH REVENUE & LEAKAGE ---
sales_month  delivered_orders  net_revenue  canceled_leakage  prev_month_rev  mom_growth_pct
    2010-12              1550    775717.50          17612.81             NaN             NaN
    2011-01              1081    670439.46          91525.05       775717.50          -13.57
    2011-02              1093    507866.54           9144.20       670439.46          -24.25
    2011-03              1440    689841.84          10621.14       507866.54           35.83
    2011-04              1236    515499.66          33316.06       689841.84          -25.27
    2011-05              1668    740036.33           9367.78       515499.66           43.56
    2011-06              1525    737683.99          14339.35       740036.33           -0.32
    2011-07              1452    688252.67          11407.26       737683.99           -6.70
    2011-08              1340    735370.22          23725.77       688252.67            6.85
    2011-09        

In [17]:
# --- SQL 2: Pareto (80/20) Customer Revenue Concentration (Window Running Total) ---
sql_2_pareto = """
WITH Customer_Spend AS (
    SELECT
        customer_segment,
        COUNT(customer_id) AS customer_count,
        ROUND(SUM(monetary_value), 2) AS segment_revenue
    FROM dim_customers_rfm
    GROUP BY customer_segment
)
SELECT
    customer_segment,
    customer_count,
    ROUND(customer_count * 100.0 / SUM(customer_count) OVER (), 2) AS pct_of_customers,
    segment_revenue,
    ROUND(segment_revenue * 100.0 / SUM(segment_revenue) OVER (), 2) AS pct_of_total_revenue,
    ROUND(
        SUM(segment_revenue) OVER (ORDER BY segment_revenue DESC) * 100.0 /
        SUM(segment_revenue) OVER (), 2
    ) AS cumulative_rev_pct
FROM Customer_Spend
ORDER BY segment_revenue DESC;
"""

In [18]:
print("\n--- SQL 2: PARETO RFM SEGMENT REVENUE CONCENTRATION ---")
print(pd.read_sql(sql_2_pareto, conn).to_string(index=False))


--- SQL 2: PARETO RFM SEGMENT REVENUE CONCENTRATION ---
   customer_segment  customer_count  pct_of_customers  segment_revenue  pct_of_total_revenue  cumulative_rev_pct
          Champions            1267             29.18       6694090.86                 76.62               76.62
    Loyal Customers             843             19.42       1035346.50                 11.85               88.47
Potential Loyalists             942             21.70        653928.56                  7.48               95.95
  At-Risk / Churned             858             19.76        254060.04                  2.91               98.86
     Need Attention             432              9.95         99801.68                  1.14              100.00


In [19]:
# --- SQL 3: Top Products by Cancellation Leakage Rate (Having + Joins) ---
sql_3_leakage = """
SELECT
    f.stock_code,
    p.product_name,
    ROUND(SUM(f.gross_amount), 2) AS gross_billed,
    ROUND(SUM(f.revenue_leakage), 2) AS canceled_amount,
    ROUND(SUM(f.revenue_leakage) * 100.0 / SUM(f.gross_amount), 2) AS leakage_rate_pct
FROM fact_transactions f
JOIN dim_products p ON f.stock_code = p.stock_code
GROUP BY f.stock_code, p.product_name
HAVING gross_billed > 15000
ORDER BY canceled_amount DESC
LIMIT 8;
"""

In [20]:
print("\n--- SQL 3: HIGH-VALUE PRODUCTS WITH HIGHEST CANCELLATION LEAKAGE ---")
print(pd.read_sql(sql_3_leakage, conn).to_string(index=False))


--- SQL 3: HIGH-VALUE PRODUCTS WITH HIGHEST CANCELLATION LEAKAGE ---
stock_code                       product_name  gross_billed  canceled_amount  leakage_rate_pct
     23843        PAPER CRAFT , LITTLE BIRDIE     336939.20        168469.60             50.00
     23166     MEDIUM CERAMIC TOP STORAGE JAR     159180.56         77479.64             48.67
     22423           REGENCY CAKESTAND 3 TIER     183853.59          9697.05              5.27
    85123A WHITE HANGING HEART T-LIGHT HOLDER     111087.05          6624.30              5.96
     21108 FAIRY CAKE FLANNEL ASSORTED COLOUR      24647.70          6591.42             26.74
     48185                 DOORMAT FAIRY CAKE      28427.81          4554.90             16.02
     21175        GIN + TONIC DIET METAL SIGN      30805.30          3775.33             12.26
     22273               FELTCRAFT DOLL MOLLY      15514.24          3512.65             22.64


In [21]:
# --- SQL 4: Top 2 Revenue-Generating Markets (Excluding UK) & Their Top Product (DENSE_RANK) ---
sql_4_country_top_prod = """
WITH Country_Product_Rank AS (
    SELECT
        f.country,
        p.product_name,
        ROUND(SUM(f.net_revenue), 2) AS product_revenue,
        DENSE_RANK() OVER (PARTITION BY f.country ORDER BY SUM(f.net_revenue) DESC) AS prod_rank
    FROM fact_transactions f
    JOIN dim_products p ON f.stock_code = p.stock_code
    WHERE f.country != 'United Kingdom' AND f.order_status = 'Delivered'
    GROUP BY f.country, p.product_name
)
SELECT country, product_name, product_revenue, prod_rank
FROM Country_Product_Rank
WHERE prod_rank <= 2
ORDER BY product_revenue DESC
LIMIT 10;
"""

In [22]:
print(
    "\n--- SQL 4: TOP SELLING PRODUCTS PER INTERNATIONAL MARKET (DENSE_RANK) ---"
)
print(pd.read_sql(sql_4_country_top_prod, conn).to_string(index=False))


--- SQL 4: TOP SELLING PRODUCTS PER INTERNATIONAL MARKET (DENSE_RANK) ---
    country                       product_name  product_revenue  prod_rank
Netherlands                 RABBIT NIGHT LIGHT          9568.48          1
    Germany           REGENCY CAKESTAND 3 TIER          9061.95          1
Netherlands ROUND SNACK BOXES SET OF4 WOODLAND          7991.40          2
       EIRE           REGENCY CAKESTAND 3 TIER          7793.25          1
     France                 RABBIT NIGHT LIGHT          7277.20          1
      Japan                 RABBIT NIGHT LIGHT          6100.32          1
       EIRE      3 TIER CAKE TIN RED AND CREAM          4265.55          2
      Spain  PINK 3 PIECE POLKADOT CUTLERY SET          3957.75          1
      Japan  ROUND SNACK BOXES SET OF 4 FRUITS          3812.10          2
      Spain  BLUE 3 PIECE POLKADOT CUTLERY SET          3671.15          2


In [23]:
df_sql.to_csv('fact_transactions_bi.csv', index=False)
dim_customers_sql.to_csv('dim_customers_rfm_bi.csv', index=False)
dim_products.to_csv('dim_products_bi.csv', index=False)

conn.close()
print("\nDone! Exported 3 Star-Schema CSVs for Power BI.")


Done! Exported 3 Star-Schema CSVs for Power BI.
